<a href="https://colab.research.google.com/github/abdulkaiyum19/Artificial-Intelligence-Driven-Multi-Cancer-Early-Detection-Using-Liquid-Biopsy-Data/blob/main/SVM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.svm import SVC, SVR
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, mean_squared_error, mean_absolute_error, r2_score

# ==================== 1. LOAD AND EXPLORE DATA ====================
print("="*80)
print("CANCER CLASSIFICATION ANALYSIS")
print("="*80)

data = pd.read_csv("Data_Sets.csv")

print("\n📊 DATASET OVERVIEW:")
print("-"*40)
print(f"Shape: {data.shape}")
print(f"Columns: {list(data.columns)}")

print("\n🔍 FIRST 5 ROWS:")
print("-"*40)
print(data.head())

print("\n📈 DATASET INFO:")
print("-"*40)
print(data.info())

print("\n📊 CLASS DISTRIBUTION:")
print("-"*40)
class_counts = data.iloc[:, -1].value_counts()
print(class_counts)
print(f"\nTotal unique classes: {len(class_counts)}")

# ==================== 2. PREPROCESS DATA ====================
print("\n" + "="*80)
print("DATA PREPROCESSING")
print("="*80)

# Extract features and labels
X = data.iloc[:, 1:-1]   # gene expression values
y = data.iloc[:, -1]    # class label

print(f"\n📐 Feature matrix shape: {X.shape}")
print(f"📐 Target vector shape: {y.shape}")

# Convert to numeric and handle missing values
X = X.apply(pd.to_numeric, errors='coerce')
X = X.fillna(X.mean())

print(f"\n✅ Missing values handled:")
print(f"   NaN values in X: {X.isna().sum().sum()}")

# Encode labels
le = LabelEncoder()
y_encoded = le.fit_transform(y)
class_names = le.classes_

print(f"\n✅ Labels encoded:")
print(f"   Original classes: {len(class_names)}")
print(f"   Encoded values: {np.unique(y_encoded)}")

# Map encoded values back to original names for interpretation
class_mapping = dict(zip(y_encoded, y))
reverse_mapping = dict(zip(range(len(class_names)), class_names))

# ==================== 3. SPLIT DATA ====================
print("\n" + "="*80)
print("DATA SPLITTING")
print("="*80)

X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded,
    test_size=0.4,
    random_state=42
)

print(f"\n✅ Data split completed:")
print(f"   Training set: {X_train.shape[0]} samples")
print(f"   Test set: {X_test.shape[0]} samples")
print(f"   Features: {X_train.shape[1]}")

# ==================== 4. SCALE FEATURES ====================
print("\n" + "="*80)
print("FEATURE SCALING")
print("="*80)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("\n✅ Features scaled using StandardScaler")

# ==================== 5. SVM FOR CLASSIFICATION ====================
print("\n" + "="*80)
print("SVM CLASSIFICATION")
print("="*80)

# Train SVM classifier (for cancer type classification)
svc_model = SVC(kernel='linear', probability=True, random_state=42)
svc_model.fit(X_train_scaled, y_train)

print("\n✅ SVM Classifier trained")
print(f"   Kernel: {svc_model.kernel}")
print(f"   Classes: {len(svc_model.classes_)}")

# Make predictions
y_pred_class = svc_model.predict(X_test_scaled)
y_pred_proba = svc_model.predict_proba(X_test_scaled)

# Convert predictions back to original class names
y_test_names = [reverse_mapping[val] for val in y_test]
y_pred_names = [reverse_mapping[val] for val in y_pred_class]

# Calculate metrics for classification
accuracy = accuracy_score(y_test, y_pred_class)
print(f"\n📊 CLASSIFICATION RESULTS:")
print(f"   Accuracy: {accuracy:.4f} ({accuracy*100:.2f}%)")

# Classification report
print("\n📋 CLASSIFICATION REPORT:")
print("-"*60)
if len(class_names) <= 20:
    print(classification_report(y_test, y_pred_class, target_names=class_names))
else:
    # For many classes, show summary
    print(f"Showing summary for {len(class_names)} classes...")
    report = classification_report(y_test, y_pred_class, output_dict=True)
    print(f"   Weighted Avg Precision: {report['weighted avg']['precision']:.3f}")
    print(f"   Weighted Avg Recall: {report['weighted avg']['recall']:.3f}")
    print(f"   Weighted Avg F1-score: {report['weighted avg']['f1-score']:.3f}")

# ==================== 6. SVR FOR REGRESSION ====================
print("\n" + "="*80)
print("SVR REGRESSION")
print("="*80)

# Train SVR (treating class as continuous for comparison)
svr_model = SVR(kernel='linear')
svr_model.fit(X_train_scaled, y_train)

print("\n✅ SVR Model trained")

# Make predictions
y_pred_reg = svr_model.predict(X_test_scaled)

# Calculate regression metrics
mse = mean_squared_error(y_test, y_pred_reg)
mae = mean_absolute_error(y_test, y_pred_reg)
r2 = r2_score(y_test, y_pred_reg)

print("\n📊 REGRESSION RESULTS:")
print(f"   Mean Squared Error (MSE): {mse:.4f}")
print(f"   Mean Absolute Error (MAE): {mae:.4f}")
print(f"   R² Score: {r2:.4f}")

# ==================== 7. VISUALIZATIONS ====================
print("\n" + "="*80)
print("VISUALIZATIONS")
print("="*80)

# Set style
plt.style.use('default')
sns.set_palette("husl")

# Create a figure with multiple subplots
fig = plt.figure(figsize=(20, 16))

# 1. Class Distribution (Bar Chart)
ax1 = plt.subplot(3, 3, 1)
top_n = min(20, len(class_counts))
class_counts.head(top_n).plot(kind='bar', ax=ax1, color='skyblue', edgecolor='black')
ax1.set_title(f'Top {top_n} Cancer Type Distribution', fontweight='bold', fontsize=12)
ax1.set_xlabel('Cancer Type')
ax1.set_ylabel('Count')
ax1.tick_params(axis='x', rotation=45, labelsize=8)
ax1.grid(True, alpha=0.3, axis='y')

# Add count labels on bars
for i, v in enumerate(class_counts.head(top_n).values):
    ax1.text(i, v + 0.5, str(v), ha='center', va='bottom', fontsize=8)

# 2. Confusion Matrix
ax2 = plt.subplot(3, 3, 2)
if len(class_names) <= 15:
    cm = confusion_matrix(y_test, y_pred_class)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax2,
                xticklabels=class_names if len(class_names) <= 10 else range(len(class_names)),
                yticklabels=class_names if len(class_names) <= 10 else range(len(class_names)))
    ax2.set_title('Confusion Matrix', fontweight='bold', fontsize=12)
    ax2.set_xlabel('Predicted')
    ax2.set_ylabel('Actual')
else:
    ax2.text(0.5, 0.5, f'Too many classes ({len(class_names)})\nfor confusion matrix',
             ha='center', va='center', fontsize=12, transform=ax2.transAxes)
    ax2.set_title('Confusion Matrix\n(Too many classes to display)', fontweight='bold', fontsize=12)
    ax2.axis('off')

# 3. Actual vs Predicted (Classification)
ax3 = plt.subplot(3, 3, 3)
sample_indices = range(min(50, len(y_test)))
if len(class_names) <= 10:
    # Show actual class names for few classes
    y_test_sample = [y_test_names[i] for i in sample_indices]
    y_pred_sample = [y_pred_names[i] for i in sample_indices]

    x_pos = np.arange(len(sample_indices))
    width = 0.35

    ax3.bar(x_pos - width/2, y_test[:len(sample_indices)], width, label='Actual',
            color='blue', alpha=0.7)
    ax3.bar(x_pos + width/2, y_pred_class[:len(sample_indices)], width, label='Predicted',
            color='red', alpha=0.7)
    ax3.set_ylabel('Class Index')
else:
    # For many classes, show as scatter
    ax3.scatter(sample_indices, y_test[:len(sample_indices)], color='blue',
                alpha=0.7, s=30, label='Actual')
    ax3.scatter(sample_indices, y_pred_class[:len(sample_indices)], color='red',
                alpha=0.7, s=30, marker='x', label='Predicted')
    ax3.set_ylabel('Encoded Class Value')

ax3.set_xlabel('Sample Index')
ax3.set_title('Classification: Actual vs Predicted (First 50 samples)', fontweight='bold', fontsize=12)
ax3.legend()
ax3.grid(True, alpha=0.3)

# 4. SVR: Actual vs Predicted Values
ax4 = plt.subplot(3, 3, 4)
sample_indices = range(min(100, len(y_test)))
ax4.scatter(sample_indices, y_test[:len(sample_indices)], color='blue',
            alpha=0.7, s=20, label='Actual')
ax4.scatter(sample_indices, y_pred_reg[:len(sample_indices)], color='red',
            alpha=0.7, s=20, marker='x', label='Predicted (SVR)')
ax4.set_xlabel('Sample Index')
ax4.set_ylabel('Encoded Class Value')
ax4.set_title('SVR: Actual vs Predicted Values', fontweight='bold', fontsize=12)
ax4.legend()
ax4.grid(True, alpha=0.3)

# 5. Prediction Confidence
ax5 = plt.subplot(3, 3, 5)
max_probs = np.max(y_pred_proba, axis=1)
n_bins = min(30, len(max_probs))
ax5.hist(max_probs, bins=n_bins, color='purple', alpha=0.7, edgecolor='black')
ax5.axvline(x=0.5, color='red', linestyle='--', alpha=0.7, label='Decision Boundary')
ax5.axvline(x=np.mean(max_probs), color='green', linestyle='-', alpha=0.7,
            label=f'Mean: {np.mean(max_probs):.2f}')
ax5.set_xlabel('Maximum Class Probability')
ax5.set_ylabel('Count')
ax5.set_title('Prediction Confidence Distribution', fontweight='bold', fontsize=12)
ax5.legend()
ax5.grid(True, alpha=0.3, axis='y')

# 6. Feature Importance (SVM coefficients)
ax6 = plt.subplot(3, 3, 6)
if hasattr(svc_model, 'coef_'):
    feature_importance = np.abs(svc_model.coef_[0])
    top_n_features = min(15, len(feature_importance))
    top_indices = np.argsort(feature_importance)[-top_n_features:][::-1]

    # Get feature names
    feature_names = X.columns
    top_features = [feature_names[i][:20] + '...' if len(str(feature_names[i])) > 20
                   else str(feature_names[i]) for i in top_indices]

    colors = plt.cm.viridis(np.linspace(0.3, 0.9, top_n_features))
    ax6.barh(range(top_n_features), feature_importance[top_indices], color=colors)
    ax6.set_yticks(range(top_n_features))
    ax6.set_yticklabels(top_features, fontsize=8)
    ax6.set_xlabel('Coefficient Magnitude')
    ax6.set_title(f'Top {top_n_features} Important Features', fontweight='bold', fontsize=12)
    ax6.invert_yaxis()
else:
    ax6.text(0.5, 0.5, 'Feature Importance\nAvailable for linear kernel',
             ha='center', va='center', fontsize=12, transform=ax6.transAxes)
    ax6.set_title('Feature Importance\n(RBF kernel used)', fontweight='bold', fontsize=12)
    ax6.axis('off')

# 7. Model Performance Comparison
ax7 = plt.subplot(3, 3, 7)
models = ['SVM Classifier', 'SVR']
metrics_svm = [accuracy, 0]  # Placeholder for SVM
metrics_svr = [0, r2]        # Placeholder for SVR

x = np.arange(len(models))
width = 0.35

bars1 = ax7.bar(x - width/2, [accuracy, 0], width, label='Accuracy/R²',
                color=['blue', 'lightblue'])
bars2 = ax7.bar(x + width/2, [0, r2], width, label='Other Metrics',
                color=['lightblue', 'green'])

ax7.set_ylabel('Score')
ax7.set_title('Model Performance Comparison', fontweight='bold', fontsize=12)
ax7.set_xticks(x)
ax7.set_xticklabels(models)
ax7.legend()
ax7.grid(True, alpha=0.3, axis='y')
ax7.set_ylim([0, 1.1])

# Add values on bars
for bars in [bars1, bars2]:
    for bar in bars:
        height = bar.get_height()
        if height > 0:
            ax7.text(bar.get_x() + bar.get_width()/2, height + 0.02,
                    f'{height:.3f}', ha='center', va='bottom', fontsize=9)

# 8. Error Distribution (SVR)
ax8 = plt.subplot(3, 3, 8)
errors = y_test - y_pred_reg
n_bins = min(30, len(errors))
ax8.hist(errors, bins=n_bins, color='orange', alpha=0.7, edgecolor='black')
ax8.axvline(x=0, color='red', linestyle='--', alpha=0.7, label='Zero Error')
ax8.axvline(x=np.mean(errors), color='green', linestyle='-', alpha=0.7,
            label=f'Mean: {np.mean(errors):.2f}')
ax8.set_xlabel('Prediction Error (Actual - Predicted)')
ax8.set_ylabel('Count')
ax8.set_title('SVR Error Distribution', fontweight='bold', fontsize=12)
ax8.legend()
ax8.grid(True, alpha=0.3, axis='y')

# 9. Sample Predictions with Actual Names
ax9 = plt.subplot(3, 3, 9)
sample_size = min(10, len(y_test))
sample_indices = range(sample_size)

# Create a table-like display
cell_text = []
for i in range(sample_size):
    actual_name = y_test_names[i]
    predicted_name = y_pred_names[i]
    is_correct = "✓" if y_test[i] == y_pred_class[i] else "✗"
    confidence = max_probs[i]
    cell_text.append([actual_name[:15] + '...' if len(actual_name) > 15 else actual_name,
                     predicted_name[:15] + '...' if len(predicted_name) > 15 else predicted_name,
                     is_correct, f'{confidence:.2f}'])

# Create table
table = ax9.table(cellText=cell_text,
                  colLabels=['Actual', 'Predicted', 'Correct', 'Confidence'],
                  cellLoc='center',
                  loc='center',
                  colWidths=[0.25, 0.25, 0.15, 0.2])

table.auto_set_font_size(False)
table.set_fontsize(9)
table.scale(1, 1.5)

# Style the table
for (row, col), cell in table.get_celld().items():
    if row == 0:
        cell.set_text_props(fontweight='bold')
        cell.set_facecolor('#4CAF50')
        cell.set_text_props(color='white')
    else:
        if cell_text[row-1][2] == "✓":
            cell.set_facecolor('#d4edda')
        else:
            cell.set_facecolor('#f8d7da')

ax9.set_title(f'Sample Predictions (First {sample_size})', fontweight='bold', fontsize=12)
ax9.axis('off')

plt.suptitle('Cancer Classification Analysis - Complete Dashboard',
             fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

# ==================== 8. ADDITIONAL BAR CHARTS ====================
print("\n" + "="*80)
print("ADDITIONAL BAR CHARTS")
print("="*80)

# Create additional figures for column-wise and row-wise analysis
fig2, axes2 = plt.subplots(2, 2, figsize=(15, 12))

# 1. Column-wise Mean Expression (Top 20 genes)
ax1 = axes2[0, 0]
column_means = X.mean().sort_values(ascending=False).head(20)
column_means.plot(kind='bar', ax=ax1, color='teal', edgecolor='black')
ax1.set_title('Top 20 Genes by Mean Expression', fontweight='bold', fontsize=12)
ax1.set_xlabel('Gene')
ax1.set_ylabel('Mean Expression')
ax1.tick_params(axis='x', rotation=45, labelsize=8)
ax1.grid(True, alpha=0.3, axis='y')

# 2. Column-wise Standard Deviation
ax2 = axes2[0, 1]
column_std = X.std().sort_values(ascending=False).head(20)
column_std.plot(kind='bar', ax=ax2, color='coral', edgecolor='black')
ax2.set_title('Top 20 Genes by Expression Variability (Std Dev)', fontweight='bold', fontsize=12)
ax2.set_xlabel('Gene')
ax2.set_ylabel('Standard Deviation')
ax2.tick_params(axis='x', rotation=45, labelsize=8)
ax2.grid(True, alpha=0.3, axis='y')

# 3. Row-wise Mean Expression by Cancer Type (if not too many classes)
ax3 = axes2[1, 0]
if len(class_names) <= 20:
    # Calculate mean expression per cancer type
    data_with_class = X.copy()
    data_with_class['Cancer_Type'] = y

    mean_by_type = data_with_class.groupby('Cancer_Type').mean().mean(axis=1)
    mean_by_type.sort_values(ascending=False).head(15).plot(kind='bar', ax=ax3,
                                                            color='purple', edgecolor='black')
    ax3.set_title('Average Expression by Cancer Type (Top 15)', fontweight='bold', fontsize=12)
    ax3.set_xlabel('Cancer Type')
    ax3.set_ylabel('Mean Expression')
    ax3.tick_params(axis='x', rotation=45, labelsize=8)
    ax3.grid(True, alpha=0.3, axis='y')
else:
    ax3.text(0.5, 0.5, f'Too many cancer types ({len(class_names)})\nfor meaningful visualization',
             ha='center', va='center', fontsize=12, transform=ax3.transAxes)
    ax3.set_title('Average Expression by Cancer Type\n(Too many types)', fontweight='bold', fontsize=12)
    ax3.axis('off')

# 4. Correct vs Incorrect Predictions
ax4 = axes2[1, 1]
correct_count = np.sum(y_test == y_pred_class)
incorrect_count = len(y_test) - correct_count
categories = ['Correct', 'Incorrect']
counts = [correct_count, incorrect_count]
colors = ['#2ecc71', '#e74c3c']

bars = ax4.bar(categories, counts, color=colors, edgecolor='black')
ax4.set_title(f'Prediction Accuracy: {accuracy*100:.1f}%', fontweight='bold', fontsize=12)
ax4.set_ylabel('Count')
ax4.grid(True, alpha=0.3, axis='y')

# Add count labels on bars
for bar, count in zip(bars, counts):
    ax4.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            str(count), ha='center', va='bottom', fontsize=10)

plt.suptitle('Gene Expression Analysis - Statistical Overview',
             fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

# ==================== 9. PRINT ACTUAL CANCER TYPES ====================
print("\n" + "="*80)
print("CANCER TYPE ANALYSIS")
print("="*80)

print("\n🔬 ACTUAL CANCER TYPES IN DATASET:")
print("-"*60)

# Check if there are "healthy" or "normal" samples
healthy_keywords = ['healthy', 'normal', 'control', 'non-tumor', 'normal tissue']
healthy_samples = []
cancer_samples = []

for class_name in class_names:
    class_name_lower = str(class_name).lower()
    if any(keyword in class_name_lower for keyword in healthy_keywords):
        healthy_samples.append(class_name)
    else:
        cancer_samples.append(class_name)

print(f"\n📊 SAMPLE BREAKDOWN:")
print(f"   Total unique labels: {len(class_names)}")
print(f"   Cancer types: {len(cancer_samples)}")
print(f"   Healthy/Normal samples: {len(healthy_samples)}")

if healthy_samples:
    print(f"\n🏥 HEALTHY/NORMAL SAMPLES FOUND:")
    for sample in healthy_samples[:10]:  # Show first 10
        print(f"   • {sample}")
    if len(healthy_samples) > 10:
        print(f"   ... and {len(healthy_samples) - 10} more")

print(f"\n🎗️  CANCER TYPES IDENTIFIED:")
if len(cancer_samples) <= 20:
    for i, cancer_type in enumerate(cancer_samples, 1):
        print(f"   {i:2d}. {cancer_type}")
else:
    # Show top 20 and summary
    for i, cancer_type in enumerate(cancer_samples[:20], 1):
        print(f"   {i:2d}. {cancer_type}")
    print(f"   ... and {len(cancer_samples) - 20} more cancer types")

# Print some actual predictions with cancer names
print(f"\n🔍 SAMPLE PREDICTIONS WITH CANCER NAMES:")
print("-"*60)
print(f"{'Index':<6} {'Actual':<30} {'Predicted':<30} {'Match':<6} {'Confidence':<10}")
print("-"*85)

for i in range(min(15, len(y_test))):
    actual_name = y_test_names[i]
    predicted_name = y_pred_names[i]
    match = "✓" if y_test[i] == y_pred_class[i] else "✗"
    confidence = max_probs[i]

    # Truncate long names
    actual_display = actual_name[:28] + '...' if len(actual_name) > 28 else actual_name
    predicted_display = predicted_name[:28] + '...' if len(predicted_name) > 28 else predicted_name

    print(f"{i:<6} {actual_display:<30} {predicted_display:<30} {match:<6} {confidence:.3f}")

# ==================== 10. SAVE RESULTS ====================
print("\n" + "="*80)
print("SAVING RESULTS")
print("="*80)

# Create results dataframe
results_df = pd.DataFrame({
    'Actual_Encoded': y_test,
    'Predicted_Encoded': y_pred_class,
    'Actual_Name': y_test_names,
    'Predicted_Name': y_pred_names,
    'Is_Correct': y_test == y_pred_class,
    'Confidence': max_probs,
    'SVR_Prediction': y_pred_reg
})

# Save to CSV
results_df.to_csv('cancer_classification_results.csv', index=False)
print("✅ Results saved to 'cancer_classification_results.csv'")

# Save model metrics
metrics_df = pd.DataFrame({
    'Metric': ['SVM_Accuracy', 'SVR_MSE', 'SVR_MAE', 'SVR_R2',
               'Num_Samples', 'Num_Features', 'Num_Classes',
               'Training_Samples', 'Test_Samples'],
    'Value': [accuracy, mse, mae, r2,
              data.shape[0], X.shape[1], len(class_names),
              X_train.shape[0], X_test.shape[0]]
})
metrics_df.to_csv('model_metrics.csv', index=False)
print("✅ Metrics saved to 'model_metrics.csv'")

# Save class mapping
class_mapping_df = pd.DataFrame({
    'Encoded_Value': list(reverse_mapping.keys()),
    'Class_Name': list(reverse_mapping.values())
})
class_mapping_df.to_csv('class_mapping.csv', index=False)
print("✅ Class mapping saved to 'class_mapping.csv'")

print("\n" + "="*80)
print("ANALYSIS COMPLETE! 🎉")
print("="*80)

print(f"\n📋 SUMMARY:")
print(f"   • Dataset: {data.shape[0]} samples, {data.shape[1]-2} features")
print(f"   • Cancer types: {len(class_names)} unique labels")
print(f"   • SVM Classification Accuracy: {accuracy*100:.2f}%")
print(f"   • SVR R² Score: {r2:.4f}")
print(f"   • Healthy samples identified: {len(healthy_samples)}")

if len(healthy_samples) > 0:
    print(f"\n💡 The model can distinguish between {len(cancer_samples)} cancer types")
    print(f"   and {len(healthy_samples)} healthy/normal conditions.")
else:
    print(f"\n💡 The model classifies between {len(cancer_samples)} different cancer types.")

CANCER CLASSIFICATION ANALYSIS

📊 DATASET OVERVIEW:
----------------------------------------
Shape: (20244, 69)
Columns: ['genename', 'Healthy_10', 'Healthy_11', 'Healthy_27', 'Healthy_35-1', 'Healthy_36-1', 'Healthy_38', 'Healthy_38o', 'Healthy_39o', 'LuCa-M0_lung395', 'LuCa-M0_lung419', 'LuCa-M0_lung492', 'LuCa-M0_lung493', 'LuCa-M0_lung496', 'LuCa-M0_lung512', 'LuCa-M0_lung513', 'LuCa-M0_lung514', 'LuCa-M0_lung515', 'LuCa-M1_lung293', 'LuCa-M1_lung323', 'LuCa-M1_lung324', 'LuCa-M1_lung417', 'LuCa-M1_lung418', 'LuCa-M1_lung517', 'HCC_HCC150', 'HCC_HCC195', 'HCC_HCC237', 'HCC_HCC256', 'HCC_HCC260', 'HCC_HCC285', 'HCC_HCC290', 'HCC_HCC320', 'HCC_HCC46', 'HCC_HCC628', 'PaCa_pancreatic15', 'PaCa_pancreatic22', 'PaCa_pancreatic27', 'PaCa_pancreatic68', 'PaCa_pancreatic69', 'PaCa_pancreatic75', 'PaCa_pancreatic9', 'BrCa_BR13', 'BrCa_BR14', 'BrCa_BR5-1', 'BrCa_BR7-1', 'CoCa_colon13', 'CoCa_colon16', 'CoCa_colon17', 'CoCa_colon19', 'GaCa_stomach1', 'GaCa_stomach2', 'GaCa_stomach3', 'GaCa_sto

KeyboardInterrupt: 